In [ ]:
import sys
import os
import importlib
import time
import datetime
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text

# Add parent directory to sys.path to allow imports from sibling modules 
sys.path.append(os.path.abspath(os.path.join('..')))

# Import the database initialization module
import init_database

# Reload the module to ensure the latest changes are used
importlib.reload(init_database)

engine = init_database.init_db()

print("✅ Module init_database reloaded successfully!")

In [ ]:
# check crm_cust_info
df_cust = pd.read_sql("SELECT * FROM silver.crm_cust_info", engine)
df_cust.head()

In [ ]:
# Check for NULLs or Duplicates in Primary Key (cst_id)
print('>> Checking cst_id: NULLs & Duplicates')

pk_issues = df_cust.groupby('cst_id').size().reset_index(name='count')
pk_issues = pk_issues[(pk_issues['count'] > 1) | (pk_issues['cst_id'].isnull())]

if pk_issues.empty:
    print("✅ Pass: No duplicates or NULLs.")
else:
    print(f"❌ Fail: Found {len(pk_issues)} issues.")
    display(pk_issues)

In [ ]:
# 2. Check for Unwanted Spaces in cst_key
# Expectation: No results (Empty DataFrame)

print('>> Checking for unwanted spaces in cst_key...')


space_issues = df_cust[df_cust['cst_key'] != df_cust['cst_key'].str.strip()]

if space_issues.empty:
    print("✅ Quality Pass: All cst_key values are properly trimmed.")
else:
    print(f"❌ Quality Fail: Found {len(space_issues)} records with leading/trailing spaces.")
    display(space_issues[['cst_id', 'cst_key']])

In [ ]:
# Check for Data Standardization & Consistency (cst_marital_status)
print('>> Checking cst_marital_status: Distinct Values')

distinct_values = df_cust['cst_marital_status'].unique()
print(f"Result: {distinct_values}")

# Validate against expected values
expected = ['Single', 'Married', 'n/a']
if all(val in expected for val in distinct_values):
    print("✅ Pass: All values are standardized.")
else:
    print("⚠️ Warning: Unexpected values found.")

In [ ]:
# check crm_prd_info
df_prd_info = pd.read_sql("SELECT * FROM silver.crm_prd_info", engine)
df_prd_info.head()

In [ ]:
# Check for NULLs or Duplicates in Primary Key (prd_id)
print('>> Checking prd_id: NULLs & Duplicates')

prd_pk_issues = df_prd_info.groupby('prd_id').size().reset_index(name='count')
prd_pk_issues = prd_pk_issues[(prd_pk_issues['count'] > 1) | (prd_pk_issues['prd_id'].isnull())]

if prd_pk_issues.empty:
    print("✅ Pass: No duplicates or NULLs.")
else:
    print(f"❌ Fail: Found {len(prd_pk_issues)} issues.")
    display(prd_pk_issues)

In [ ]:
# Check for unwanted spaces in prd_nm
print('>> Checking prd_nm: Unwanted Spaces')

prd_nm_issues = df_prd_info[df_prd_info['prd_nm'] != df_prd_info['prd_nm'].str.strip()]

if prd_nm_issues.empty:
    print("✅ Pass: No leading/trailing spaces.")
else:
    print(f"❌ Fail: Found {len(prd_nm_issues)} records.")
    display(prd_nm_issues[['prd_id', 'prd_nm']])

In [ ]:
# Check for NULLs or Negative values in prd_cost
print('>> Checking prd_cost: NULLs & Negatives')

cost_issues = df_prd_info[(df_prd_info['prd_cost'] < 0) | (df_prd_info['prd_cost'].isnull())]

if cost_issues.empty:
    print("✅ Pass: All costs are valid.")
else:
    print(f"❌ Fail: Found {len(cost_issues)} invalid records.")
    display(cost_issues[['prd_id', 'prd_cost']])

In [ ]:
# Check for Data Standardization (prd_line)
print('>> Checking prd_line: Distinct Values')

distinct_lines = df_prd_info['prd_line'].unique()
print(f"Result: {distinct_lines}")

# Optional: Validate against expected values
expected_lines = ['Road', 'Mountain', 'Other Sales', 'Touring', 'n/a']
if all(val in expected_lines for val in distinct_lines):
    print("✅ Pass: All values are standardized.")
else:
    print("⚠️ Warning: Unexpected categories found.")

In [ ]:
# Check for Invalid Date Orders (Start Date > End Date)
print('>> Checking Date Order: prd_start_dt vs prd_end_dt')

# Note: We filter out NaT in end_dt as it represents active products
date_issues = df_prd_info[df_prd_info['prd_end_dt'] < df_prd_info['prd_start_dt']]

if date_issues.empty:
    print("✅ Pass: All date ranges are logical.")
else:
    print(f"❌ Fail: Found {len(date_issues)} records where Start > End.")
    display(date_issues[['prd_id', 'prd_start_dt', 'prd_end_dt']])

In [ ]:
# check crm_sales_details
df_sales = pd.read_sql("SELECT * FROM silver.crm_sales_details", engine)
df_sales.head()

In [ ]:
# Check for invalid dates (Out of range or non-standard)
print('>> Checking sls_due_dt: Date Range Validation')

# Checking if dates fall outside a reasonable range (1900-2050)
invalid_dates = df_sales[
    (df_sales['sls_due_dt'].dt.year < 1900) | 
    (df_sales['sls_due_dt'].dt.year > pd.Timestamp.now().year) | 
    (df_sales['sls_due_dt'].isnull())
]

if invalid_dates.empty:
    print("✅ Pass: All dates are within valid range.")
else:
    print(f"❌ Fail: Found {len(invalid_dates)} invalid dates.")
    display(invalid_dates[['sls_ord_num', 'sls_due_dt']])

In [ ]:
# Check for invalid date sequences (Order Date > Ship/Due Date)
print('>> Checking Date Order: sls_order_dt vs Ship/Due dates')

date_seq_issues = df_sales[
    (df_sales['sls_order_dt'] > df_sales['sls_ship_dt']) | 
    (df_sales['sls_order_dt'] > df_sales['sls_due_dt'])
]

if date_seq_issues.empty:
    print("✅ Pass: All date sequences are logical.")
else:
    print(f"❌ Fail: Found {len(date_seq_issues)} records with inconsistent dates.")
    display(date_seq_issues[['sls_ord_num', 'sls_order_dt', 'sls_ship_dt', 'sls_due_dt']])

In [ ]:
# Check Data Consistency: Sales = Quantity * Price
print('>> Checking Sales Consistency: (Sales = Qty * Price) & Positivity')

# Using np.isclose to avoid floating point precision errors during comparison
math_issues = df_sales[
    ~np.isclose(df_sales['sls_sales'], df_sales['sls_quantity'] * df_sales['sls_price']) |
    (df_sales['sls_sales'] <= 0) | 
    (df_sales['sls_quantity'] <= 0) | 
    (df_sales['sls_price'] <= 0) |
    (df_sales['sls_sales'].isnull())
]

if math_issues.empty:
    print("✅ Pass: Sales calculations and values are consistent.")
else:
    print(f"❌ Fail: Found {len(math_issues)} records with calculation or value issues.")
    display(math_issues[['sls_ord_num', 'sls_sales', 'sls_quantity', 'sls_price']].head())

In [ ]:
# check erp_cust_az12
df_erp_cust = pd.read_sql("SELECT * FROM silver.erp_cust_az12", engine)
df_erp_cust.head()

In [ ]:
# Check for out-of-range birthdates (1924 to Today)
print('>> Checking BDATE: Range Validation')

today = pd.Timestamp.today()
bdate_issues = df_erp_cust[(df_erp_cust['BDATE'] < '1924-01-01') | (df_erp_cust['BDATE'] > today)]

if bdate_issues.empty:
    print("✅ Pass: All birthdates are within a valid range.")
else:
    print(f"❌ Fail: Found {len(bdate_issues)} out-of-range dates.")
    display(bdate_issues[['CID', 'BDATE']])

In [ ]:
# Check for Data Standardization (GEN)
print('>> Checking GEN: Distinct Values')

distinct_gen = df_erp_cust['GEN'].unique()
print(f"Result: {distinct_gen}")

# Expected values after cleaning: Male, Female, n/a
expected_gen = ['Male', 'Female', 'n/a']
if all(val in expected_gen for val in distinct_gen):
    print("✅ Pass: Gender values are standardized.")
else:
    print(f"⚠️ Warning: Found unexpected gender values: {[v for v in distinct_gen if v not in expected_gen]}")

In [ ]:
# check erp_loc_a101
df_erp_loc = pd.read_sql("SELECT * FROM silver.erp_loc_a101", engine)
df_erp_loc['CNTRY'].unique()

In [ ]:
# Check for Country Standardization (CNTRY)
print('>> Checking CNTRY: Distinct Values')

# Getting unique countries sorted for better visibility
distinct_countries = sorted(df_erp_loc['CNTRY'].unique())
print(f"Result: {distinct_countries}")

# Expected full country names (Example based on previous cleaning logic)
expected_countries = [
    'Australia', 'Canada', 'France', 'Germany', 
    'United Kingdom', 'United States', 'n/a'
]

# Identify any values that were not correctly mapped
unexpected = [c for c in distinct_countries if c not in expected_countries]

if not unexpected:
    print("✅ Pass: All countries are properly standardized.")
else:
    print(f"⚠️ Warning: Found unmapped or unexpected values: {unexpected}")

In [ ]:
# check erp_px_cat_g1v2
df_erp_px = pd.read_sql("SELECT * FROM silver.erp_px_cat_g1v2", engine) 
df_erp_px.head()


In [ ]:
# Check for unwanted spaces in category fields
print('>> Checking Category Fields: Unwanted Spaces')

# Identifying records where any field doesn't match its stripped version
space_issues_erp = df_erp_px[
    (df_erp_px['CAT'] != df_erp_px['CAT'].str.strip()) | 
    (df_erp_px['SUBCAT'] != df_erp_px['SUBCAT'].str.strip()) | 
    (df_erp_px['MAINTENANCE'] != df_erp_px['MAINTENANCE'].str.strip())
]

if space_issues_erp.empty:
    print("✅ Pass: All category fields are properly trimmed.")
else:
    print(f"❌ Fail: Found {len(space_issues_erp)} records with padding spaces.")
    display(space_issues_erp.head())

In [ ]:
# Check for Standardization in maintenance field
print('>> Checking maintenance: Distinct Values')

distinct_maint = df_erp_px['MAINTENANCE'].unique()
print(f"Result: {distinct_maint}")

# Check if there are any unexpected variations 
# (e.g., if you expected only 'Yes', 'No', 'n/a')
if len(distinct_maint) <= 3: # Assuming a small number of standard values
    print("✅ Pass: Maintenance values appear standardized.")
else:
    print(f"⚠️ Warning: Found {len(distinct_maint)} distinct values. Review for consistency.")